# Última pesquisa Focus: explorando os dados brutos

Este notebook lê o snapshot SQLite validado em `dados/atual.json` e abre os **JSON originais** guardados em `dados/bruto/`. O objetivo é entender a estrutura das respostas da [API de Expectativas de Mercado do BCB](https://dadosabertos.bcb.gov.br/dataset/expectativas-mercado), antes de montar séries ou gráficos.

Na verificação de **02/11/2026**, a última `Data` disponível nos 12 recursos estatísticos da API era **23/10/2026**. O código abaixo recalcula isso ao ser executado. `Data` é a data da observação; ela não deve ser confundida com a publicação do PDF semanal.

**Limite dos dados:** “brutos” aqui significa a resposta JSON original da API de estatísticas, sem transformação pelo projeto. Cada objeto contém medidas como `Media`, `Mediana`, `Minimo`, `Maximo`, `DesvioPadrao` e, quando disponível, `numeroRespondentes`; **não é uma previsão individual de um respondente**. O BCB lista o recurso `ExpectativasMercadoInstituicoes` como desativado por risco de quebra de confidencialidade da autoria dos microdados ([catálogo oficial](https://dadosabertos.bcb.gov.br/dataset/expectativas-mercado/resource/2231aecd-3dbe-4e1a-bae6-674d3be225a1)).


## 1. Abrir a versão local

Execute o notebook a partir da raiz do projeto ou da pasta `notebooks/`. O banco é consultado seletivamente; não carregaremos os quase cinco milhões de registros na memória.

In [ ]:
projeto <- if (file.exists("dados/atual.json")) "." else if (file.exists("../dados/atual.json")) ".." else stop("Abra o notebook na raiz do projeto ou em notebooks/")
if (requireNamespace("renv", quietly = TRUE)) renv::load(project = normalizePath(projeto), quiet = TRUE)
suppressPackageStartupMessages({library(DBI); library(RSQLite); library(jsonlite); library(digest)})
atual <- fromJSON(file.path(projeto, "dados/atual.json"))
con <- dbConnect(SQLite(), file.path(projeto, atual$banco))
inventario <- fromJSON(file.path(projeto, atual$manifestos, "inventario.json"), simplifyVector = FALSE)
cat("Snapshot:", atual$coleta, "\nBanco:", atual$banco, "\nRecursos:", length(inventario), "\n")

## 2. Qual é a última observação em cada recurso?

Os 12 recursos estatísticos possuem `Data`. O 13º, `DatasReferencia`, tem outro significado e será visto separadamente. A tabela mostra quantas linhas e indicadores existem na última data de cada recurso.

In [ ]:
estatisticos <- setdiff(names(inventario), "DatasReferencia")
resumo <- do.call(rbind, lapply(estatisticos, function(recurso) {
  tabela <- as.character(dbQuoteIdentifier(con, recurso))
  indicador <- if ("Indicador" %in% names(inventario[[recurso]]$campos)) "Indicador" else "indicador"
  ultima <- dbGetQuery(con, sprintf("SELECT max(Data) AS data FROM %s", tabela))$data[[1]]
  n <- dbGetQuery(con, sprintf("SELECT count(*) AS linhas, count(DISTINCT %s) AS indicadores FROM %s WHERE Data = ?", dbQuoteIdentifier(con, indicador), tabela), params = list(ultima))
  data.frame(recurso = recurso, ultima_data = ultima, linhas = n$linhas, indicadores = n$indicadores)
}))
data_focus <- max(resumo$ultima_data)
stopifnot(length(unique(resumo$ultima_data)) == 1L)
resumo

In [ ]:
rotulos <- c(
  ExpectativaMercadoMensais = "Mensais",
  ExpectativasMercadoSelic = "Selic",
  ExpectativasMercadoTrimestrais = "Trimestrais",
  ExpectativasMercadoAnuais = "Anuais",
  ExpectativasMercadoInflacao12Meses = "Inflação 12 meses",
  ExpectativasMercadoInflacao24Meses = "Inflação 13 a 24 meses",
  ExpectativasMercadoTop5Mensais = "Top 5 mensais",
  ExpectativasMercadoTop5Selic = "Top 5 Selic",
  ExpectativaMercadoTop5Trimestral = "Top 5 trimestral",
  ExpectativasMercadoTop5Anuais = "Top 5 anuais",
  ExpectativasMercadoTop5Inflacao12Meses = "Top 5 inflação 12m",
  ExpectativasMercadoTop5Inflacao24Meses = "Top 5 inflação 13–24m"
)
ordem <- order(resumo$linhas)
options(repr.plot.width = 10, repr.plot.height = 6)
op <- par(no.readonly = TRUE)
par(mar = c(5, 16, 4, 2), bg = "white", fg = "#334155",
    col.axis = "#334155", col.lab = "#334155", col.main = "#0F172A")
barplot(resumo$linhas[ordem], names.arg = rotulos[resumo$recurso[ordem]],
        horiz = TRUE, las = 1, col = "#0F766E", border = NA,
        xlab = "Linhas na última data", main = paste("Volume por recurso em", data_focus),
        cex.names = 0.82)
par(op)

## 3. Encontrar e abrir as páginas JSON originais

Vamos usar o recurso anual agregado. A coluna `coleta` no SQLite indica qual extração forneceu os registros da data. O manifesto dessa extração aponta para as páginas comprimidas e seus hashes SHA-256. Uma data pode ocupar mais de uma página; por isso a busca considera todas as páginas cujo intervalo inclui `data_focus`.

In [ ]:
recurso <- "ExpectativasMercadoAnuais"
origens <- dbGetQuery(con, sprintf("SELECT DISTINCT coleta FROM %s WHERE Data = ?", dbQuoteIdentifier(con, recurso)), params = list(data_focus))$coleta
stopifnot(length(origens) == 1L)
coleta_bruta <- origens[[1]]
pasta_manifestos <- file.path(projeto, "dados/manifestos", coleta_bruta, recurso)
arquivos_manifesto <- list.files(pasta_manifestos, pattern = "\\.json$", full.names = TRUE)
paginas <- do.call(rbind, lapply(arquivos_manifesto, function(arquivo) {
  m <- fromJSON(arquivo, simplifyVector = FALSE)
  selecionadas <- Filter(function(p) !is.null(p$data_min) && !is.null(p$data_max) &&
                           p$data_min <= data_focus && p$data_max >= data_focus, m$paginas)
  if (!length(selecionadas)) return(NULL)
  do.call(rbind, lapply(selecionadas, function(p)
    data.frame(janela = m$janela$id, arquivo = p$arquivo, linhas_pagina = p$linhas,
               data_min = p$data_min, data_max = p$data_max, sha256 = p$sha256, url = p$url)))
}))
stopifnot(nrow(paginas) > 0L)
paginas[, c("janela", "arquivo", "linhas_pagina", "data_min", "data_max", "sha256")]

In [ ]:
caminhos <- file.path(projeto, "dados/bruto", coleta_bruta, recurso, paginas$janela, paginas$arquivo)
hashes <- vapply(caminhos, function(p) digest(file = p, algo = "sha256"), character(1))
stopifnot(identical(unname(hashes), paginas$sha256))
ler_json_gz <- function(arquivo) {
  comprimido <- readBin(arquivo, "raw", n = file.info(arquivo)$size)
  fromJSON(rawToChar(memDecompress(comprimido, type = "gzip")), simplifyVector = FALSE)
}
respostas <- lapply(caminhos, ler_json_gz)
cat("Chaves do documento JSON:", paste(names(respostas[[1]]), collapse = ", "), "\n")
cat("URL da primeira página:", paginas$url[[1]], "\n")
str(respostas[[1]]$value[[1]], max.level = 1)

In [ ]:
linhas_brutas <- do.call(c, lapply(respostas, function(r)
  Filter(function(linha) identical(linha$Data, data_focus), r$value)))
linhas_sqlite <- dbGetQuery(con, sprintf("SELECT count(*) AS n FROM %s WHERE Data = ?", dbQuoteIdentifier(con, recurso)), params = list(data_focus))$n[[1]]
stopifnot(length(linhas_brutas) == linhas_sqlite)
data.frame(data = data_focus, linhas_json = length(linhas_brutas),
           linhas_sqlite = linhas_sqlite, paginas_conferidas = length(caminhos),
           hashes_ok = all(hashes == paginas$sha256))

## 4. Esquema: campos nativos, tipos e chaves

O inventário foi derivado dos metadados OData. Ele preserva a grafia e o tipo originais. `recurso` e `coleta` são as duas colunas acrescentadas no SQLite. As famílias **não têm exatamente as mesmas colunas**: observe, por exemplo, a diferença de maiúsculas/minúsculas entre os recursos anuais e o Top 5 Selic.

In [ ]:
dicionario <- do.call(rbind, lapply(names(inventario), function(recurso) {
  campos <- unlist(inventario[[recurso]]$campos, use.names = TRUE)
  data.frame(recurso = recurso, campo = names(campos), tipo_odata = unname(campos),
             na_chave = names(campos) %in% unlist(inventario[[recurso]]$chave))
}))
dicionario[dicionario$recurso %in% c("ExpectativasMercadoAnuais",
                                    "ExpectativasMercadoTop5Selic", "DatasReferencia"), ]

In [ ]:
data.frame(recurso = names(inventario),
           colunas_nativas = vapply(inventario, function(x) length(x$campos), integer(1)),
           chave_candidata = vapply(inventario, function(x) paste(unlist(x$chave), collapse = " + "), character(1)))

## 5. Por que há várias linhas para um mesmo indicador?

Uma linha anual é identificada por indicador, possível detalhe, `Data`, ano projetado (`DataReferencia`) e `baseCalculo`. A mediana é **uma estatística daquela combinação**, não uma série única de IPCA. O exemplo a seguir mantém a mesma observação e mostra anos e bases distintos.

In [ ]:
ano_pesquisa <- as.integer(format(as.Date(data_focus), "%Y"))
anos_exemplo <- as.character(ano_pesquisa + 0:2)
ano_alvo <- as.character(ano_pesquisa + 1L)
dbGetQuery(con, "
  SELECT Indicador, IndicadorDetalhe, Data, DataReferencia, baseCalculo,
         Mediana, numeroRespondentes
  FROM ExpectativasMercadoAnuais
  WHERE Data = ? AND Indicador = ? AND DataReferencia IN (?,?,?)
  ORDER BY DataReferencia, baseCalculo",
  params = c(list(data_focus, "IPCA"), as.list(anos_exemplo)))

In [ ]:
top5 <- dbGetQuery(con, "
  SELECT Indicador, Data, DataReferencia, tipoCalculo, Media, Mediana
  FROM ExpectativasMercadoTop5Anuais
  WHERE Data = ? AND Indicador = ? AND DataReferencia = ?
  ORDER BY tipoCalculo",
  params = list(data_focus, "IPCA", ano_alvo))
top5

`tipoCalculo` é uma dimensão nativa do Top 5 e não deve ser misturada entre linhas. O Top 5 também não oferece sempre os mesmos campos de respondentes do agregado.

## 6. Comparar um registro bruto com o SQLite

Escolhemos **IPCA projetado para o ano seguinte à última observação, base 0**. O JSON preserva inclusive campos `null`; o SQLite guarda o número para consulta. A comparação abaixo usa os valores originais, sem arredondar.

In [ ]:
candidatos <- Filter(function(x) identical(x$Indicador, "IPCA") &&
  is.null(x$IndicadorDetalhe) && identical(x$DataReferencia, ano_alvo) &&
  identical(as.integer(x$baseCalculo), 0L), linhas_brutas)
stopifnot(length(candidatos) == 1L)
bruto <- candidatos[[1]]
sql <- dbGetQuery(con, "
  SELECT Media, Mediana, DesvioPadrao, Minimo, Maximo, numeroRespondentes
  FROM ExpectativasMercadoAnuais
  WHERE Data = ? AND Indicador = 'IPCA' AND IndicadorDetalhe IS NULL
    AND DataReferencia = ? AND baseCalculo = 0", params = list(data_focus, ano_alvo))
stopifnot(nrow(sql) == 1L)
campos_numericos <- names(sql)
comparacao <- data.frame(campo = campos_numericos,
  valor_json = vapply(campos_numericos, function(n) as.numeric(bruto[[n]]), numeric(1)),
  valor_sqlite = vapply(campos_numericos, function(n) as.numeric(sql[[n]][[1]]), numeric(1)))
comparacao$iguais <- mapply(function(a,b) isTRUE(all.equal(a,b,tolerance=0)),
                             comparacao$valor_json, comparacao$valor_sqlite)
stopifnot(all(comparacao$iguais))
comparacao

## 7. Ausências e o recurso de datas de referência

`NULL` no JSON é ausência, nunca zero. As colunas numéricas disponíveis variam por recurso. A tabela abaixo conta ausências **somente na última data** do agregado anual; o relatório histórico completo está em `ausentes_historicos.csv`.

In [ ]:
numericas <- names(Filter(function(tipo) tipo %in% c("Edm.Decimal", "Edm.Int32"),
                         inventario[[recurso]]$campos))
faltantes <- do.call(rbind, lapply(numericas, function(campo) {
  q <- sprintf("SELECT count(*) AS ausentes FROM %s WHERE Data = ? AND %s IS NULL",
               dbQuoteIdentifier(con, recurso), dbQuoteIdentifier(con, campo))
  data.frame(campo = campo, ausentes = dbGetQuery(con, q, params = list(data_focus))$ausentes[[1]])
}))
faltantes

In [ ]:
cat("Campos de DatasReferencia:", paste(names(inventario$DatasReferencia$campos), collapse = ", "), "\n")
dbGetQuery(con, "SELECT * FROM DatasReferencia ORDER BY DataReferencia1 DESC, Indicador LIMIT 8")

`DatasReferencia` ajuda a interpretar os rankings Top 5. `DataReferencia1` e `DataReferencia2` não substituem a `Data` da observação estatística. Algumas datas de referência podem estar no futuro em relação à última pesquisa.

## 8. Próximas explorações

Troque o nome do recurso e examine `inventario[[recurso]]$campos` e `inventario[[recurso]]$chave` antes de consultar. Para uma série temporal, mantenha fixos indicador, período projetado, base/tipo de cálculo e estatística; varie apenas `Data`. Os gráficos reproduzíveis do projeto estão em `analises/graficos_exemplo.R`.

Ao terminar, feche a conexão.

In [ ]:
dbDisconnect(con)
cat("Conexão SQLite fechada. Snapshot explorado:", atual$coleta, "\n")